# Nemotron Run-All Notebook

This notebook provides a single, end-to-end workflow for:
- loading configuration from environment variables,
- preparing training/evaluation data,
- training a LoRA adapter,
- saving artifacts, and
- creating `submission.zip` in the repository root.

Expected final artifact: **`submission.zip`**.

## Prerequisites
- Python environment with project dependencies from `pyproject.toml`.
- Training data at `DATA_DIR/train.csv` (default `./data/raw/nvidia-nemotron-model-reasoning-challenge/train.csv`).
- GPU is recommended for full training.


## 1) Set Up Notebook Environment
Import core libraries, set reproducibility options, and configure notebook display/output behavior for development.

In [ ]:
from __future__ import annotations

import json
import os
import random
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch

try:
    from dotenv import load_dotenv
    load_dotenv(override=False)
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 120)
print("Environment ready")
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2) Define Project Configuration
Centralize paths and training parameters so the workflow can be controlled from environment variables.

In [ ]:
def env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


def env_float(name: str, default: float) -> float:
    raw = os.getenv(name)
    return float(raw) if raw is not None else default


def env_int(name: str, default: int) -> int:
    raw = os.getenv(name)
    return int(raw) if raw is not None else default


@dataclass
class RunConfig:
    model_name: str
    data_dir: Path
    train_csv: Path
    output_dir: Path
    learning_rate: float
    batch_size: int
    epochs: int
    max_length: int
    seed: int
    lora_r: int
    lora_alpha: int
    lora_dropout: float
    target_modules: List[str]
    load_in_4bit: bool
    load_in_8bit: bool
    cpu_offload: bool
    eval_ratio: float
    run_training: bool


def load_config() -> RunConfig:
    data_dir = Path(os.getenv("DATA_DIR", "./data/raw/nvidia-nemotron-model-reasoning-challenge"))
    train_csv_default = data_dir / "train.csv"
    train_csv = Path(os.getenv("NEMOTRON_TRAIN_CSV", str(train_csv_default)))

    target_modules_raw = os.getenv("NEMOTRON_TARGET_MODULES", "out_proj")
    target_modules = [part.strip() for part in target_modules_raw.split(",") if part.strip()]

    cfg = RunConfig(
        model_name=os.getenv("NEMOTRON_MODEL_NAME", "nvidia/Llama-3.1-Nemotron-Nano-4B-v1.1"),
        data_dir=data_dir,
        train_csv=train_csv,
        output_dir=Path(os.getenv("NEMOTRON_OUTPUT_DIR", "./artifacts/nemotron-lora")),
        learning_rate=env_float("NEMOTRON_LEARNING_RATE", 2e-4),
        batch_size=env_int("NEMOTRON_BATCH_SIZE", 1),
        epochs=env_int("NEMOTRON_EPOCHS", 1),
        max_length=env_int("NEMOTRON_MAX_LENGTH", 1024),
        seed=env_int("NEMOTRON_SEED", 42),
        lora_r=env_int("NEMOTRON_LORA_R", 16),
        lora_alpha=env_int("NEMOTRON_LORA_ALPHA", 32),
        lora_dropout=env_float("NEMOTRON_LORA_DROPOUT", 0.05),
        target_modules=target_modules,
        load_in_4bit=env_bool("NEMOTRON_LOAD_IN_4BIT", True),
        load_in_8bit=env_bool("NEMOTRON_LOAD_IN_8BIT", False),
        cpu_offload=env_bool("NEMOTRON_CPU_OFFLOAD", False),
        eval_ratio=env_float("NEMOTRON_EVAL_RATIO", 0.1),
        run_training=env_bool("NEMOTRON_RUN_TRAINING", True),
    )

    if cfg.load_in_4bit and cfg.load_in_8bit:
        raise ValueError("Only one of NEMOTRON_LOAD_IN_4BIT and NEMOTRON_LOAD_IN_8BIT can be true")
    if cfg.lora_r > 32:
        raise ValueError("NEMOTRON_LORA_R cannot exceed 32")
    if cfg.eval_ratio <= 0 or cfg.eval_ratio >= 1:
        raise ValueError("NEMOTRON_EVAL_RATIO must be between 0 and 1")

    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    return cfg


config = load_config()
print(json.dumps({
    "model_name": config.model_name,
    "train_csv": str(config.train_csv),
    "output_dir": str(config.output_dir),
    "load_in_4bit": config.load_in_4bit,
    "load_in_8bit": config.load_in_8bit,
    "cpu_offload": config.cpu_offload,
    "epochs": config.epochs,
    "batch_size": config.batch_size,
    "run_training": config.run_training,
}, indent=2))

## 3) Implement Core Functions
Implement minimal data preparation, prompt formatting, dataset objects, model loading, training, and packaging functions.

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


def load_dataframe(train_csv: Path) -> pd.DataFrame:
    if not train_csv.exists():
        raise FileNotFoundError(f"Training CSV not found: {train_csv}")
    df = pd.read_csv(train_csv)
    required = {"id", "prompt", "response", "target"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    return df


def split_dataframe(df: pd.DataFrame, eval_ratio: float, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_df, eval_df = train_test_split(df, test_size=eval_ratio, random_state=seed, stratify=df["target"])
    return train_df.reset_index(drop=True), eval_df.reset_index(drop=True)


def build_training_text(prompt: str, response: str, target: int) -> str:
    label = "yes" if int(target) == 1 else "no"
    return (
        "### Prompt:\n"
        f"{str(prompt).strip()}\n\n"
        "### Response:\n"
        f"{str(response).strip()}\n\n"
        "### Classification:\n"
        f"{label}"
    )


class SFTDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_length: int):
        self.samples = []
        for _, row in frame.iterrows():
            full_text = build_training_text(row["prompt"], row["response"], row["target"])
            prompt_only = (
                "### Prompt:\n"
                f"{str(row['prompt']).strip()}\n\n"
                "### Response:\n"
                f"{str(row['response']).strip()}\n\n"
                "### Classification:\n"
            )

            tokenized = tokenizer(full_text, truncation=True, max_length=max_length, padding=False)
            prompt_tokens = tokenizer(prompt_only, truncation=True, max_length=max_length, padding=False)

            labels = tokenized["input_ids"].copy()
            prompt_len = min(len(prompt_tokens["input_ids"]), len(labels))
            labels[:prompt_len] = [-100] * prompt_len

            self.samples.append({
                "input_ids": tokenized["input_ids"],
                "attention_mask": tokenized["attention_mask"],
                "labels": labels,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        return {k: torch.tensor(v, dtype=torch.long) for k, v in self.samples[idx].items()}


def load_model_and_tokenizer(cfg: RunConfig):
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = None
    if cfg.load_in_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif cfg.load_in_8bit:
        quantization_config = BitsAndBytesConfig(load_in_8bit=True)

    model_kwargs = {
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }

    if quantization_config is not None:
        model_kwargs["quantization_config"] = quantization_config
        model_kwargs["device_map"] = "auto"
    elif torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch.bfloat16

    if cfg.cpu_offload:
        model_kwargs["device_map"] = "auto"

    try:
        model = AutoModelForCausalLM.from_pretrained(cfg.model_name, **model_kwargs)
    except Exception as exc:
        print(f"Primary model load failed: {exc}")
        fallback_kwargs = {
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
            "device_map": "auto",
        }
        model = AutoModelForCausalLM.from_pretrained(cfg.model_name, **fallback_kwargs)

    if cfg.load_in_4bit or cfg.load_in_8bit:
        model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=cfg.target_modules,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    return model, tokenizer


def train_adapter(cfg: RunConfig, model, tokenizer, train_df: pd.DataFrame, eval_df: pd.DataFrame):
    train_dataset = SFTDataset(train_df, tokenizer, cfg.max_length)
    eval_dataset = SFTDataset(eval_df, tokenizer, cfg.max_length)

    args = TrainingArguments(
        output_dir=str(cfg.output_dir),
        learning_rate=cfg.learning_rate,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        num_train_epochs=cfg.epochs,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        bf16=torch.cuda.is_available(),
        fp16=False,
        remove_unused_columns=False,
        report_to="none",
        seed=cfg.seed,
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()
    return trainer


def save_adapter(model, tokenizer, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    config_file = output_dir / "adapter_config.json"
    if not config_file.exists():
        raise FileNotFoundError(f"Missing required file: {config_file}")
    return output_dir


def package_submission(adapter_dir: Path, zip_path: Path) -> Path:
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Adapter directory not found: {adapter_dir}")

    required = [adapter_dir / "adapter_config.json"]
    for req in required:
        if not req.exists():
            raise FileNotFoundError(f"Required adapter file missing: {req}")

    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(adapter_dir.iterdir()):
            if path.is_file():
                zf.write(path, arcname=path.name)

    with zipfile.ZipFile(zip_path, "r") as zf:
        names = set(zf.namelist())
        if "adapter_config.json" not in names:
            raise ValueError("Invalid archive: adapter_config.json missing")

    return zip_path

## 4) Create a Minimal Execution Pipeline
Compose the functions into a single workflow that can be called with one line.

In [ ]:
def run_pipeline(cfg: RunConfig) -> dict:
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg.seed)

    df = load_dataframe(cfg.train_csv)
    train_df, eval_df = split_dataframe(df, cfg.eval_ratio, cfg.seed)
    model, tokenizer = load_model_and_tokenizer(cfg)

    if cfg.run_training:
        trainer = train_adapter(cfg, model, tokenizer, train_df, eval_df)
        train_metrics = trainer.state.log_history[-1] if trainer.state.log_history else {}
    else:
        train_metrics = {"skipped_training": True}

    adapter_dir = save_adapter(model, tokenizer, cfg.output_dir)
    zip_path = package_submission(adapter_dir, Path.cwd() / "submission.zip")

    return {
        "rows_total": len(df),
        "rows_train": len(train_df),
        "rows_eval": len(eval_df),
        "adapter_dir": str(adapter_dir),
        "zip_path": str(zip_path),
        "train_metrics": train_metrics,
    }

## 5) Run a First End-to-End Test
Run the complete pipeline and inspect key outputs.

In [ ]:
result = run_pipeline(config)
result

## 6) Add Basic Assertions and Debug Output
Validate assumptions and print concise diagnostics for fast iteration.

In [ ]:
zip_path = Path(result["zip_path"])
adapter_dir = Path(result["adapter_dir"])

assert zip_path.exists(), f"Expected zip artifact at {zip_path}"
assert adapter_dir.exists(), f"Expected adapter directory at {adapter_dir}"
assert (adapter_dir / "adapter_config.json").exists(), "adapter_config.json is missing"

with zipfile.ZipFile(zip_path, "r") as zf:
    names = sorted(zf.namelist())
    assert "adapter_config.json" in names, "submission.zip missing adapter_config.json"

print("✅ Pipeline completed")
print("rows_total:", result["rows_total"])
print("rows_train:", result["rows_train"])
print("rows_eval:", result["rows_eval"])
print("adapter_dir:", adapter_dir)
print("submission_zip:", zip_path)
print("zip_contents_sample:", names[:10])